# R3maJ - GigaLearnCPP RL Bot (Colab)

Run this notebook on a **GPU runtime** (Runtime > Change runtime type > T4 GPU) for training speed, or CPU (slow) to sanity-check the build.

Workflow:
1. Mount Google Drive
2. Unpack `R3maJ_colab_upload.zip` (place it in `My Drive/R3maJ/` first)
3. Install build tools + libtorch
4. Build the trainer
5. **(Auto)** restore the latest checkpoint from Drive
6. Start training with **automatic checkpoint backup to My Drive** every sync interval

## 0) Mount Google Drive (click the link, authorize, copy the code back)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- CONFIG ----
GDRIVE_BASE = '/content/drive/MyDrive/R3maJ'          # folder on My Drive
ZIP_NAME    = 'R3maJ_colab_upload.zip'                 # must be in GDRIVE_BASE
WORK        = '/content/R3maJ'                         # unpacked project here
BUILD_DIR   = WORK + '/build'                          # cmake build dir
OUT_DIR     = WORK + '/build/Release'                  # binary lands here (VS generator style)
OUT_DIR_LIN = WORK + '/build/bin'                      # ...or here (Unix Makefiles style)
CKPT_LOCAL  = OUT_DIR + '/checkpoints'                 # trainer's local save dir
CKPT_DRIVE  = GDRIVE_BASE + '/checkpoints'             # backup target on My Drive
SYNC_SECS   = 120                                     # backup interval
DEVICE      = 'cuda'                                   # 'cuda' (T4 GPU) or 'cpu'
NUM_GAMES   = 96                                       # parallel games
PHASE_IDX   = 0                                       # 0..3

import os, glob, time, shutil, subprocess, threading, sys, json
print('Config loaded.')
print('exist check:', ZIP_NAME, '->', os.path.exists(GDRIVE_BASE + '/' + ZIP_NAME))

In [ ]:
# If the zip isn't on Drive yet, upload it straight from here
if not os.path.exists(GDRIVE_BASE + '/' + ZIP_NAME):
    from google.colab import files
    print('Upload the file', ZIP_NAME, 'now:')
    up = files.upload()
    if ZIP_NAME in up:
        os.makedirs(GDRIVE_BASE, exist_ok=True)
        with open(os.path.join(GDRIVE_BASE, ZIP_NAME), 'wb') as f:
            f.write(up[ZIP_NAME])
        print('Saved to', GDRIVE_BASE + '/' + ZIP_NAME)
    else:
        print('Uploaded something else; expected filename:', ZIP_NAME)

## 1) Unpack the project

In [ ]:
if os.path.exists(WORK):
    !rm -rf {WORK}
os.makedirs(WORK, exist_ok=True)
!unzip -q {GDRIVE_BASE}/{ZIP_NAME} -d {WORK}
print('Unpacked. Project skeleton:')
!ls {WORK}

## 2) Install build dependencies

Needed: `cmake`, `g++`, `git`, `python3-dev` (libpython for the Python-embed link), and `pip`-installed **torch** (CPU or CUDA) that provides `libtorch` for the C++ build.

_T4 (free) has compute capability 7.5. If you get an L4/A100/V100 the `-DCMAKE_CUDA_ARCHITECTURES=75` fix forces a fast build; see the build cell note._

In [ ]:
# Ubuntu build tools + python dev headers (libpython for the Python-embed link)
!apt-get -q update
!apt-get -q install -y cmake g++ git python3-dev libx11-dev
print('apt done')
!cmake --version | head -1
!g++ --version | head -1

In [ ]:
import torch
pref = os.path.dirname(torch.__file__)
print('torch prefix:', pref)
print('TorchConfig exists:', os.path.exists(os.path.join(pref, 'share', 'cmake', 'Torch', 'TorchConfig.cmake')))
print('CUDA built into torch:', torch.version.cuda)
print('torch version:', torch.__version__)
# Resolve symlinks so find_package(Torch) sees real libtorch .so files
_lib = os.path.join(pref, 'lib')
for _n in os.listdir(_lib):
    _p = os.path.join(_lib, _n)
    if os.path.islink(_p):
        _t = os.readlink(_p)
        _abs = _t if os.path.isabs(_t) else os.path.join(_lib, _t)
        if os.path.exists(_abs) and not os.path.exists(_p):
            _real = os.path.realpath(_abs)
            if os.path.exists(_real):
                os.remove(_p)
                shutil.copy2(_real, _p)
print('libtorch .so symlinks resolved (real files in place)')

## 3) Configure and build

Uses CMake's `TORCH_INSTALL_PREFIX` (same trick as the Windows build) so `find_package(Torch)` finds pip-installed torch.

- **GPU runtime**: `CMAKE_CUDA_ARCHITECTURES` is force-set to `75` in the CMakeLists. T4 = 75, so it Just Works. On another GPU (L4=89, A100=80, V100=70, K80=37) this builds for 75 and Torch CUDA handles it via PTX. If you hit CUDA `no kernel image` errors at runtime, edit `CMakeLists.txt` `CMAKE_CUDA_ARCHITECTURES` to `80;89;90` (or `70;75;80`) and reconfigure.
- **CPU runtime**: nvcc is absent, so it falls back to a CPU-only build automatically.

In [ ]:
pref = os.path.dirname(torch.__file__)
os.makedirs(BUILD_DIR, exist_ok=True)
cfg = [
    'cmake', '-S', WORK, '-B', BUILD_DIR,
    '-DTORCH_INSTALL_PREFIX=' + pref,
    '-DCMAKE_BUILD_TYPE=Release',
    '-DCMAKE_EXPORT_COMPILE_COMMANDS=ON',
]
print('Configure:\n', ' '.join(cfg))
r = subprocess.run(cfg, capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-2000:])
if r.returncode != 0:
    raise SystemExit('cmake configure FAILED (see output above)')

In [ ]:
print('Building (this takes several minutes)...')
r = subprocess.run(['cmake', '--build', BUILD_DIR, '--config', 'Release', '-j', str(os.cpu_count())],
                   capture_output=True, text=True)
tail = (r.stdout or '')[-4000:] + (r.stderr or '')[-2000:]
print(tail)
if r.returncode != 0:
    raise SystemExit('build FAILED (see output above)')
print('Build OK.')

In [ ]:
# Locate the trainer binary
exe = None
for cand in glob.glob(OUT_DIR + '/R3maJ') + glob.glob(OUT_DIR_LIN + '/R3maJ'):
    if os.path.isfile(cand):
        exe = cand
        break
# Copy DLLs / python scripts / collision meshes next to the binary if needed
bin_dir = os.path.dirname(exe) if exe else OUT_DIR
os.makedirs(bin_dir, exist_ok=True)
python_scripts_src = os.path.join(WORK, 'thirdparty', 'GigaLearnCPP-Leak', 'GigaLearnCPP', 'python_scripts')
if os.path.isdir(python_scripts_src):
    shutil.copytree(python_scripts_src, os.path.join(bin_dir, 'python_scripts'), dirs_exist_ok=True)
if os.path.isdir(os.path.join(WORK, 'collision_meshes')):
    shutil.copytree(os.path.join(WORK, 'collision_meshes'), os.path.join(bin_dir, 'collision_meshes'), dirs_exist_ok=True)
print('Binary:', exe)
if not exe:
    print('NOT FOUND — build output check:')
    !find {WORK}/build -maxdepth 3 -name 'R3maJ' -type f

## 4) Restore latest checkpoint from My Drive (auto-resume)

Copies any previously backed-up numbered checkpoint folders back into the local `checkpoints/` dir; the learner auto-loads the highest. Trains fresh if none exist.

In [ ]:
os.makedirs(CKPT_LOCAL, exist_ok=True)
if os.path.isdir(CKPT_DRIVE):
    n = 0
    for d in sorted(os.listdir(CKPT_DRIVE)):
        s = os.path.join(CKPT_DRIVE, d)
        d_ = os.path.join(CKPT_LOCAL, d)
        if d.isdigit() and os.path.isdir(s) and not os.path.exists(d_):
            shutil.copytree(s, d_)
            n += 1
    print(f'Restored {n} checkpoint(s) from Drive.')
else:
    print('No checkpoints on Drive yet — starting fresh.')
print('Local checkpoints:', sorted([d for d in os.listdir(CKPT_LOCAL) if d.isdigit()], key=int) if os.path.isdir(CKPT_LOCAL) else [])

## 5) Train + automatic checkpoint backup to My Drive

Runs `R3maJ` in the foreground. A background thread **backs up completed checkpoint folders to `My Drive/R3maJ/checkpoints/`** every `SYNC_SECS`, keeping them in sync right after each save.

> Colab VMs recycle after ~12h. When it disconnects, just re-run cells 4+5 with the same Drive — it resumes from the latest checkpoint.

**Stop training:** interrupt the cell (runtime stop button), then run cell 6 with `ONLY_SYNC=True` to flush the final checkpoint to Drive.

In [ ]:
# ---- checkpoint backup engine ----
import pathlib

def _safe_copy_dir(src, dst):
    """Copy a fully-written numbered checkpoint dir. Retries a few times if mid-save."""
    for _ in range(5):
        if not os.path.isdir(src):
            return False
        try:
            if os.path.exists(dst):
                shutil.rmtree(dst, ignore_errors=True)
            shutil.copytree(src, dst)
            return True
        except Exception:
            time.sleep(2)
    return False

def sync_checkpoints(force=False):
    if not os.path.isdir(CKPT_LOCAL):
        return 0
    os.makedirs(CKPT_DRIVE, exist_ok=True)
    copied = 0
    dirs = [d for d in os.listdir(CKPT_LOCAL) if d.isdigit()]
    dirs.sort(key=int)
    for d in dirs:
        src = os.path.join(CKPT_LOCAL, d)
        dst = os.path.join(CKPT_DRIVE, d)
        # skip dirs still being written (mtime newer than sync interval)
        try:
            if not force and (time.time() - os.path.getmtime(src)) < SYNC_SECS:
                continue
        except OSError:
            continue
        # skip identical copies
        if os.path.isdir(dst) and glob.glob(dst + '/*') and \
           os.path.getmtime(dst) >= os.path.getmtime(src):
            continue
        if _safe_copy_dir(src, dst):
            copied += 1
    return copied

def _backup_loop(stop):
    while not stop.is_set():
        try:
            n = sync_checkpoints()
            if n:
                print(f'[backup] uploaded {n} checkpoint(s) -> {CKPT_DRIVE}', flush=True)
        except Exception as e:
            print('[backup] error:', e, flush=True)
        time.sleep(SYNC_SECS)
    sync_checkpoints(force=True)

print('Backup engine ready. (sync table: local={} drive={} interval={}s)'.format(CKPT_LOCAL, CKPT_DRIVE, SYNC_SECS))

In [ ]:
import signal

ONLY_SYNC = False   # set True to skip training and just flush checkpoints to Drive

if ONLY_SYNC:
    sync_checkpoints(force=True)
    print('Checkpoints synced. Done.')
else:
    stop = threading.Event()
    t = threading.Thread(target=_backup_loop, args=(stop,), daemon=True)
    t.start()
    print('Backup thread started.')

    env = dict(os.environ)
    cmd = [exe, '--device', DEVICE, '--save-dir', CKPT_LOCAL, '--phase', str(PHASE_IDX), '--games', str(NUM_GAMES)]
    print('Running:', ' '.join(cmd))
    try:
        proc = subprocess.Popen(cmd, cwd=bin_dir, env=env)
        proc.wait()
    except KeyboardInterrupt:
        proc.send_signal(signal.SIGINT)
        proc.wait(timeout=60)
        print('Stopped train; flushing checkpoint...')
        sync_checkpoints(force=True)
        stop.set()
        print('Final sync done. Goodbye.')